# scikit-learn dataset demo

Two standard scikit-learn datasets in `DetailViewsWidget`, both projected to 2-D with PCA:

1. **digits** with the digit class as label and the 8×8 images shown in the detail views
   (`image_shape=(8, 8)`);
2. **iris** with the species as label and the default tabular detail views.

The detail-view type is selected by `dataset_type`: `"default"` gives tabular insets; `image_shape`
gives image insets for any grid and expects one column per pixel keyed `"{row}x{col}"` (1-based),
grayscale 0–255 (`dataset_type="mnist"` is the fixed 28×28 preset of the same thing; the app's own
`chess` / `rubik` / `cctv` / `gymnasium` conventions also exist). Neither dataset has trajectories,
so the widget shows the points and their labels; the kNN graph and HDBSCAN clustering are computed
inside the widget.

Requires `scikit-learn`, `numpy`, `pandas` in the kernel's environment plus the `detailviews` wheel
(see `simple_widget_demo.ipynb` for the setup script).

## Digits as image insets

The 8×8 digits (values 0–16) are scaled to 0–255 and passed as they are, one column per pixel keyed
`"1x1"` to `"8x8"`; `image_shape=(8, 8)` tells the widget the grid, and the insets upscale it with
nearest neighbour.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from detailviews import DetailViewsWidget

digits = load_digits()
xy = PCA(n_components=2, random_state=0).fit_transform(digits.data)

imgs = np.rint(digits.images * (255.0 / 16.0)).astype(int)   # (1797, 8, 8) → 0..255

pixel_cols = {f"{r + 1}x{c + 1}": imgs[:, r, c] for r in range(8) for c in range(8)}
df = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "digit": digits.target.astype(str), **pixel_cols})

DetailViewsWidget.from_dataframe(df, x="x", y="y", label="digit", image_shape=(8, 8))

## Iris via the record-list constructor

The same idea without pandas: iris (150 rows, 4 features), species as label, default tabular
detail views over the four measurements.

In [8]:
from sklearn.datasets import load_iris

iris = load_iris()
xy_iris = PCA(n_components=2, random_state=0).fit_transform(iris.data)
rows = [
    {
        "x": float(px), "y": float(py),
        "species": iris.target_names[t],
        **{name.replace(" (cm)", "").replace(" ", "_"): float(v)
           for name, v in zip(iris.feature_names, feats)},
    }
    for (px, py), t, feats in zip(xy_iris, iris.target, iris.data)
]

DetailViewsWidget(
    data=rows,
    columnMapping={"x": "x", "y": "y", "label": "species"},
    datasetType="default",
)